In [1]:
import sys
from tokenizers import Tokenizer
from typing import Optional, Tuple
import torch

sys.path.append('..')

tokenizer = Tokenizer.from_file("tokenizer.json")
train = ''
test = ''

with open('train.txt') as f:
    train += f.read()

with open('test.txt') as f:
    test += f.read()

In [2]:
encoded_train = tokenizer.encode(train)
encoded_test = tokenizer.encode(test)

print(len(encoded_train),len(encoded_test))

66256749 8254662


In [3]:
train_data = torch.tensor(encoded_train.ids, dtype=torch.long)
val_data = torch.tensor(encoded_test.ids, dtype=torch.long)

In [5]:
from torch.utils.data import DataLoader, Dataset
from typing import Tuple

class TextDataset(Dataset):

    def __init__(self, data: torch.Tensor, block_size:int) -> None:
        super().__init__()
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size
    
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.data[index: index+self.block_size], self.data[index+1: index+self.block_size+1]
    

def get_data_loader(
        train_data: torch.Tensor,
        val_data: torch.Tensor,
        block_size: int,
        batch_size: int,
        device: torch.device,
        multi_gpu: Optional[bool] = False,
) ->  Tuple[DataLoader, DataLoader]:
    # if multi_gpu:
    #     print("Using multiple GPUs")
    #     train_data = torch.nn.DataParallel(train_data)
    #     val_data = torch.nn.DataParallel(val_data)
    train_dataset = TextDataset(train_data.to(device=device), block_size=block_size)
    val_dataset = TextDataset(val_data.to(device=device), block_size=block_size)


    training_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    validation_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return training_loader, validation_loader


In [6]:
from typing import Dict

def calculate_loss(
        model: torch.nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        steps: int
) -> Dict[str, float]:
    output = {}
    for split, loader in [('train', train_loader), ('val', val_loader)]:

        losses = torch.zeros(steps +1)

        for i, (x,y) in enumerate(loader):
            if i > steps:
                break

            with torch.no_grad():
                _, loss = model(x, y)

            losses[i] = loss.item()
        output[split] = losses.mean().item()
    
    model.train()
    return output

def save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    filename: str = 'checkpoint.pth'
) -> None:
    checkpoint = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, filename)

In [7]:
from gpt_model import GPTModel

torch.manual_seed(1024)
torch.cuda.empty_cache()

block_size = 256
embedding_size = 128
num_heads = 8
num_layers = 6
batch_size = 256
vocab_size = tokenizer.get_vocab_size()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Vocab Size: {vocab_size}")

model = GPTModel(
    vocab_size=vocab_size,
    block_size=block_size,
    embedding_size=embedding_size,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=0.2,
    device=device,
    multi_gpu=torch.cuda.device_count() > 1
)
# if torch.cuda.device_count() > 1:
#   print("Let's use", torch.cuda.device_count(), "GPUs!")
#   model = torch.nn.DataParallel(model)
model = model.to(device)

Using device: cuda:0
Vocab Size: 1376


In [8]:
max_steps = 100
eval_interval = 10
eval_steps = 200
learning_rate =1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
training_loader, validation_loader = get_data_loader(
    train_data=train_data,
    val_data=val_data,
    block_size=block_size,
    batch_size=batch_size,
    device=device,
    multi_gpu=torch.cuda.device_count() > 1
)

train_losses = []
val_losses = []

print("Training Started")

for i in range(max_steps):
    for batch_id, (x,y) in enumerate(training_loader):
        if batch_id % eval_interval == 0 or batch_id == len(training_loader) - 1:
            losses = calculate_loss(
                model=model,
                train_loader=training_loader,
                val_loader= validation_loader,
                steps=max(eval_steps, len(validation_loader))
            )
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])

            print(f"itr:{i}: step {batch_id+1}/{len(training_loader)} training loss: {losses['train']:.4f}, validation loss: {losses['val']:.4f}")

        logits, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=i,
        loss=loss.item(),
        filename=f"checkpoint_{i}.pth"
    )


Training Started
itr:0: step 1/258815 training loss: 7.3037, validation loss: 7.3030
itr:0: step 11/258815 training loss: 6.5292, validation loss: 6.5287
itr:0: step 21/258815 training loss: 6.3057, validation loss: 6.3050
itr:0: step 31/258815 training loss: 6.0821, validation loss: 6.0813
itr:0: step 41/258815 training loss: 5.8439, validation loss: 5.8429
itr:0: step 51/258815 training loss: 5.6064, validation loss: 5.6052
itr:0: step 61/258815 training loss: 5.3781, validation loss: 5.3768
itr:0: step 71/258815 training loss: 5.1590, validation loss: 5.1576
itr:0: step 81/258815 training loss: 4.9475, validation loss: 4.9457
itr:0: step 91/258815 training loss: 4.7444, validation loss: 4.7423
itr:0: step 101/258815 training loss: 4.5518, validation loss: 4.5494
itr:0: step 111/258815 training loss: 4.3717, validation loss: 4.3689
itr:0: step 121/258815 training loss: 4.2056, validation loss: 4.2025
itr:0: step 131/258815 training loss: 4.0549, validation loss: 4.0513
itr:0: step 14

KeyboardInterrupt: 